In [3]:
!pip install rdkit pandas numpy

In [4]:
from rdkit import Chem
from rdkit.Chem import rdRGroupDecomposition, AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
import pandas as pd

In [8]:
df = pd.read_csv("egfr_features_processed.csv")

print(df.head())
print(df.columns)

   action_type  activity_comment  activity_id activity_properties  \
0          NaN               NaN      32260.0                  []   
1          NaN               NaN      32263.0                  []   
2          NaN               NaN      32265.0                  []   
3          NaN               NaN      32267.0                  []   
4          NaN               NaN      32270.0                  []   

  assay_chembl_id                                  assay_description  \
0    CHEMBL674637  Inhibitory activity towards tyrosine phosphory...   
1    CHEMBL621151  Inhibition of autophosphorylation of human epi...   
2    CHEMBL615325  Inhibition of ligand-induced proliferation in ...   
3    CHEMBL674637  Inhibitory activity towards tyrosine phosphory...   
4    CHEMBL621151  Inhibition of autophosphorylation of human epi...   

  assay_type  assay_variant_accession  assay_variant_mutation bao_endpoint  \
0          B                      NaN                     NaN  BAO_0000190

In [10]:

# ─── 3A: Murcko Scaffold Extraction ───────────────────────────────────────────
def get_murcko_scaffold(smiles):
    """
    Extract Bemis-Murcko scaffold — the ring system + linkers.
    In patent terms: the 'core' of a Markush general formula.
    """
    if pd.isna(smiles):
        return None

    smiles = str(smiles)

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)

df = df.dropna(subset=['canonical_smiles'])

df['canonical_smiles'] = df['canonical_smiles'].astype(str)

print(df['canonical_smiles'].head())

df['murcko_scaffold'] = df['canonical_smiles'].apply(get_murcko_scaffold)

# Scaffold frequency → patent claim breadth proxy
scaffold_freq = df['murcko_scaffold'].value_counts().reset_index()
scaffold_freq.columns = ['scaffold', 'compound_count']
top_scaffolds = scaffold_freq.head(10)
print("Top 10 scaffolds (Markush core families):")
print(top_scaffolds)

# ─── 3B: SMARTS-Based R-Group Zone Detection ──────────────────────────────────
# Quinazoline core — the defining scaffold of EGFR inhibitors
# (Erlotinib, Gefitinib, Afatinib all share this)
QUINAZOLINE_SMARTS = "c1cnc2ccccc2n1"   # simplified quinazoline
ANILINE_SMARTS     = "c1ccc(N)cc1"       # common R-group attachment point

def detect_pharmacophore_zones(smiles, smarts_list):
    """
    Check which pharmacophore patterns are present.
    Analogous to identifying R-group positions in a Markush formula.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {}
    results = {}
    for name, smarts in smarts_list.items():
        pattern = Chem.MolFromSmarts(smarts)
        results[name] = mol.HasSubstructMatch(pattern) if pattern else False
    return results

pharmacophores = {
    'quinazoline_core':  QUINAZOLINE_SMARTS,
    'aniline_rgroup':    ANILINE_SMARTS,
    'pyrimidine_core':   "c1ncnc(N)n1",
    'acrylamide_warhead': "C=CC(=O)N",   # covalent EGFR inhibitors (Afatinib, Osimertinib)
}

pharm_df = pd.DataFrame([
    detect_pharmacophore_zones(s, pharmacophores)
    for s in df['canonical_smiles']
], index=df.index)

df = pd.concat([df, pharm_df], axis=1)

# ─── 3C: RDKit R-Group Decomposition (True Markush Analysis) ──────────────────
def rgroup_decompose(smiles_list, core_smarts):
    """
    Decompose a compound series against a Markush core.
    Returns R-group SMILES for each position — exactly what patent analysts do.
    """
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    mols = [m for m in mols if m is not None]
    core = Chem.MolFromSmarts(core_smarts)

    groups, unmatched = rdRGroupDecomposition.RGroupDecompose(
        [core], mols, asSmiles=True, asRows=True
    )
    rgroup_df = pd.DataFrame(groups)
    print(f"Decomposed {len(rgroup_df)} compounds | "
          f"Unmatched: {len(unmatched)}")
    return rgroup_df

# Use quinazoline core for EGFR series decomposition
quinazoline_core_smarts = "[#6]1:[#7]:[#6]:[#7]:[#6]2:[#6]:[#6]:[#6]:[#6]:[#6]:1:2"
egfr_subset = df[df['quinazoline_core'] == True]['canonical_smiles'].head(100)

if len(egfr_subset) > 5:
    rg_df = rgroup_decompose(egfr_subset.tolist(), quinazoline_core_smarts)
    print(f"R-group positions identified: {list(rg_df.columns)}")

0    Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...
1    Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...
2    Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...
3    Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...
4    Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...
Name: canonical_smiles, dtype: object
Top 10 scaffolds (Markush core families):
                                            scaffold  compound_count
0                                           c1ccccc1              67
1                          c1ccc(Nc2ncnc3ccccc23)cc1              65
2                          c1ccc(Nc2ncnc3cnccc23)cc1              40
3                          c1ccc(Nc2ncnc3ccncc23)cc1              32
4                          c1ccc(Nc2ccnc3ccccc23)cc1              31
5                     c1ccc(Nc2ncnc3c2sc2ccccc23)cc1              27
6  O=C(Nc1ccccc1)c1c(SSc2[nH]c3ccccc3c2C(=O)Nc2cc...              25
7                  c1ccc(Nc2ncnc3cc4[nH]ccc4cc23)cc1              24
8              c1ccc(Nc2

In [11]:
# Save processed dataframe for next notebook
df.to_csv("egfr_markush_processed.csv", index=False)

print("File saved successfully")

File saved successfully
